# Migration to GRIB2 - practical 1
## Statistically processed parameters and the new timespan keyword

Many forecast parameters are a statistic over a time window: the maximum 2 m temperature in the last 3 hours,
the maximum wind gust in the last 6 hours, the precipitation accumulated since the start of the forecast...

- In **IFS Cycle 50r1** data, every window has its own paramId: `mx2t3` (last 3 hours) and `mx2t6` (last 6 hours) are two
  different parameters.
- In **full GRIB2**, **one paramId** identifies the parameter *and* the statistic (maximum, minimum, mean,
  accumulation...), and the length of the window is stored separately. ecCodes exposes it as **`timespan`**,
  and MARS requests use the new **`timespan`** keyword.

| Case | `timespan` | Parameters |
| --- | --- | --- |
| 1 | `none` | 2 m temperature |
| 2 | `1h` / `3h` / `6h` | maximum and minimum 2 m temperature |
| 3 | `1h` / `3h` / `6h` | maximum 10 m wind gust |
| 4 | `fs` | total precipitation |

`expver` changes only because the GRIB2 data are a test dataset. Once GRIB2 is operational, it will be
`expver=1` again.

`grid` and `area` interpolate the data to 0.25° over Europe, to keep the downloads small.

## Setup

You need **earthkit-data** (with the mars source), **earthkit-plots** and a recent **ecCodes under the hood**.  
ecCodes older than 2.42.0 cannot decode the new encoding correctly - always use the latest release.  

If you are using this outside of ECMWF HPC, you will also need ecmwf-api-client which can be installed using pip.

In [1]:
import earthkit.data as ekd

# The keys we print for every field
keys = ["metadata.edition", "metadata.shortName", "metadata.paramId", "metadata.units",
        "metadata.stepRange", "metadata.timespan", "metadata.name"]

## `timespan=1h/3h/6h`: maximum and minimum 2 m temperature


### GRIB1
In GRIB1 every window is a separate parameter: **six paramIds**.

| GRIB1 `param` | GRIB2 `param` | GRIB2 `timespan` | step range at step 24 |
| --- | --- | --- | --- |
| 201 (`mx2t`) | 237167 (`max_2t`) | `1h` | 23-24 |
| 228026 (`mx2t3`) | 237167 (`max_2t`) | `3h` | 21-24 |
| 121 (`mx2t6`) | 237167 (`max_2t`) | `6h` | 18-24 |
| 202 (`mn2t`) | 238167 (`min_2t`) | `1h` | 23-24 |
| 228027 (`mn2t3`) | 238167 (`min_2t`) | `3h` | 21-24 |
| 122 (`mn2t6`) | 238167 (`min_2t`) | `6h` | 18-24 |

The fields are the same (same step ranges, same units); what changes is **how you ask for them**.

- Hourly output is only produced up to step 90, so `timespan=1h` is not availabe for steps above 90.  
- 3-hourly output is produced up to step 144, so `timespan=3h` is not availabe for steps above 144.  
- 6-hourly output is available for all the steps 0 to 360, so there are no limits for using `timespan=6h`

In [2]:
t2m_maxmin_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "1",                # changed
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "201/228026/121/202/228027/122",    # Also shotNames: mx2t/mx2t3/mx2t6/mn2t/mn2t3/mn2t6
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [3]:
t2m_maxmin_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,1,mx2t,201,K,23-24,None,Maximum temperature at 2 metres since previous...
1,1,mx2t3,228026,K,21-24,None,Maximum temperature at 2 metres in the last 3 ...
2,1,mx2t6,121,K,18-24,None,Maximum temperature at 2 metres in the last 6 ...
3,1,mn2t,202,K,23-24,None,Minimum temperature at 2 metres since previous...
4,1,mn2t3,228027,K,21-24,None,Minimum temperature at 2 metres in the last 3 ...
5,1,mn2t6,122,K,18-24,None,Minimum temperature at 2 metres in the last 6 ...


### GRIB2

**Two paramIds**, one for Tmin one for Tmax. Time window will be specified using the `timespan` parameter.

In [4]:
t2m_maxmin_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "2250",             # changed (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "237167/238167",     # changed, shortNames: max_2t/min_2t
        "timespan": "1h/3h/6h",       # NEW: 1, 3 and 6 hours
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [5]:
t2m_maxmin_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,2,max_2t,237167,K,23-24,1h,Time-maximum 2 metre temperature
1,2,min_2t,238167,K,23-24,1h,Time-minimum 2 metre temperature
2,2,max_2t,237167,K,21-24,3h,Time-maximum 2 metre temperature
3,2,min_2t,238167,K,21-24,3h,Time-minimum 2 metre temperature
4,2,max_2t,237167,K,18-24,6h,Time-maximum 2 metre temperature
5,2,min_2t,238167,K,18-24,6h,Time-minimum 2 metre temperature


## `timespan=1h/3h/6h`: maximum 10 m wind gust

### GRIB1

In [6]:
gust_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "1",                # changed
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "49/228028/123",     # changed: 10fg/10fg3/10fg6
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [7]:
gust_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,1,10fg,49,m s**-1,23-24,None,Maximum 10 metre wind gust since previous post...
1,1,10fg3,228028,m s**-1,21-24,None,Maximum 10 metre wind gust in the last 3 hours
2,1,10fg6,123,m s**-1,18-24,None,Maximum 10 metre wind gust in the last 6 hours


### GRIB2

In [8]:
gust_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "2250",             # changed (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "237318",            # changed: max_i10fg
        "timespan": "1h/3h/6h",       # NEW: previous 1, 3 and 6 hours
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [9]:
gust_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,2,max_i10fg,237318,m s**-1,23-24,1h,Time-maximum 10 metre wind gust
1,2,max_i10fg,237318,m s**-1,21-24,3h,Time-maximum 10 metre wind gust
2,2,max_i10fg,237318,m s**-1,18-24,6h,Time-maximum 10 metre wind gust


| GRIB1 `param` | GRIB2 `param` | GRIB2 `timespan` |
| --- | --- | --- |
| 49 (`10fg`) | 237318 (`max_i10fg`) | `1h` |
| 228028 (`10fg3`) | 237318 (`max_i10fg`) | `3h` |
| 123 (`10fg6`) | 237318 (`max_i10fg`) | `6h` |

Three GRIB1 parameters, one GRIB2 parameter.

## `timespan=fs`: total precipitation

### GRIB1

In [ ]:
tp_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "1",                # changed (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "228",               # changed: tp in m
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [11]:
tp_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,1,tp,228,m,24,None,Total precipitation


### GRIB2

In [ ]:
tp_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "2250",             # changed (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "228228",            # changed: tp in kg m**-2
        "timespan": "fs",             # NEW: accumulated from the start of the forecast
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [13]:
tp_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,2,tp,228228,kg m**-2,0-24,fs,Total Precipitation


| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `param` | 228 | 228228 | changed |
| units | m | kg m\*\*-2 (= mm) | changed: values are 1000 times larger |
| `timespan` | - | `fs` | new |
| stepType in the data | instant | accum | changed: GRIB1 does not say it is an accumulation |
| step range in the data | 24 | 0-24 | changed: GRIB2 says the accumulation starts at step 0 |

Two changes at once: the new `timespan=fs` and the new paramId for WMO units. The shortName stays `tp`, so
check `paramId` or `units`, not just the name.

In [14]:
tp_grib1.ls(keys = ["metadata.maximum","metadata.units", "metadata.paramId"])

,metadata.maximum,metadata.units,metadata.paramId
0,0.039365,m,228


In [15]:
tp_grib2.ls(keys = ["metadata.maximum","metadata.units", "metadata.paramId"])

,metadata.maximum,metadata.units,metadata.paramId
0,39.368164,kg m**-2,228228


##  `timespan=none`: 2 m temperature

### GRIB1

In [ ]:
t2m_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "1",                # changed - 1 is operational data
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "167",               # same: 2t
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [17]:
t2m_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,1,2t,167,K,24,None,2 metre temperature


### GRIB2

In [ ]:
t2m_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "2250",             # changed (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "167",               # same: 2t
        "timespan": "none",           # NEW: instantaneous field, no time window
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [19]:
t2m_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name
0,2,2t,167,K,24,none,2 metre temperature


| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `param` | 167 | 167 | same |
| `timespan` | - | `none` | new |
| field | `2t`, instant, step 24 | `2t`, instant, step 24 | same |

Only the new `timespan=none` keyword. In the data, GRIB1 has no `timespan` key (`None`), GRIB2 has the
string `none`.  
In MARS request, parameters that have `timespan=none`, don't need this parameter explicitly in the request.

## The full list of affected parameters

| GRIB1 shortName (paramId) | full GRIB2 shortName | GRIB2 paramId | timespan | stepType |
| --- | --- | --- | --- | --- |
| `mx2t` (201), `mx2t3` (228026), `mx2t6` (121) | `max_2t` | **237167** | 1h / 3h / 6h | max |
| `mn2t` (202), `mn2t3` (228027), `mn2t6` (122) | `min_2t` | **238167** | 1h / 3h / 6h | min |
| `10fg` (49), `10fg3` (228028), `10fg6` (123) | `max_i10fg` | **237318** | 1h / 3h / 6h | max |
| `mxtpr` (228226), `mxtpr3` (228222), `mxtpr6` (228224) | `max_tprate` | **237055** | 1h / 3h / 6h | max |
| `mntpr` (228227), `mntpr3` (228223), `mntpr6` (228225) | `min_tprate` | **238055** | 1h / 3h / 6h | min |
| `mxcape6` (228035) | `max_cape` | **237287** | 6h | max |
| `mxcapes6` (228036) | `max_capes` | **237321** | 6h | max |
| `litota1` (228051), `litota3` (228057), `litota6` (228058) | `avg_litoti` | **235326** | 1h / 3h / 6h | avg |
| `ptype_sev1h/3h/6h` (260318/260319/260338) | `sev_ptype` | **260682** | 1h / 3h / 6h | severity |
| `ptype_freq1h/3h/6h` (260320/260321/260339) | `freq_ptype` | **260683** | 1h / 3h / 6h | mode |

Note that the *instantaneous* gust `i10fg` (228029) is not a time statistic and is not affected.

## Your turn now

### Practical A
Modify the requests below to download Averaged total lightning flash density (litota) paremeters. (You can read more about them [here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-Meantotallightningflashdensity))

In [20]:
lt_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "1",
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "228051/228057/228058",  # Add paramIDs here 
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [ ]:
lt_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "2250",             # (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "235326",               # Add paramIDs here
        "timespan": "1h/3h/6h",           # Add timespans you wish here
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

Explore the parameters to check if it is only timespan that has been added or something else is changing. (Hint: add some of the metadata parameters as keys in the ls)

In [22]:
lt_grib1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,litota1,2026-09-02,2026-09-01,1 days,0,entire_atmosphere,None,regular_ll
1,litota3,2026-09-02,2026-09-01,1 days,0,entire_atmosphere,None,regular_ll
2,litota6,2026-09-02,2026-09-01,1 days,0,entire_atmosphere,None,regular_ll


In [23]:
lt_grib2.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,avg_litoti,2026-09-02,2026-09-01,1 days,0,entire_atmosphere,0,regular_ll
1,avg_litoti,2026-09-02,2026-09-01,1 days,0,entire_atmosphere,0,regular_ll
2,avg_litoti,2026-09-02,2026-09-01,1 days,0,entire_atmosphere,0,regular_ll


### Practical B
Modify the requests below to download the convective precipitation paremeter. (You can read more about them [here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-ParametersencodedbyECMWFinunitsofmetresormetresofwaterequivalent))

In [24]:
cp_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "1",
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "143",  # Add paramIDs here 
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

In [25]:
cp_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",                
        "stream": "oper",             
        "type": "fc",                 
        "expver": "2250",             # changed (test data only)
        "date": "20260901",           
        "time": "00",                 
        "levtype": "sfc",             
        "param": "228143",               # Add paramIDs here
        "timespan": "fs",           # Add timespan needed for precipitation
        "step": "24",                 
        "grid": "0.25/0.25",          
        "area": "75/-30/25/45",       
    },
).to_fieldlist()

Explore the convective precipitation data in GRIB1 nd GRIB2 format.  
Compare the two. What are the units in GRIB2?

In [28]:
cp_grib1.ls(keys=keys + ["metadata.maximum","metadata.units", "metadata.paramId"])

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name,metadata.maximum
0,1,cp,143,m,24,None,Convective precipitation,0.029178


In [29]:
cp_grib2.ls(keys=keys + ["metadata.maximum","metadata.units", "metadata.paramId"])

,metadata.edition,metadata.shortName,metadata.paramId,metadata.units,metadata.stepRange,metadata.timespan,metadata.name,metadata.maximum
0,2,cp,228143,kg m**-2,0-24,fs,Convective precipitation,29.176758


Further reading: 
- [Migration to GRIB2 - Test GRIB2 data in MARS](https://confluence.ecmwf.int/pages/viewpage.action?pageId=716086411)
- [Changes in the MARS language](https://confluence.ecmwf.int/pages/viewpage.action?pageId=621051006)
- [Documentation of statistically processed paremeters over the time window](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-Parametersstatisticallyprocessedoveratimewindow)